# Sistema de Recomendação — Produtos Fitness
**Projeto Aplicado III — Universidade Presbiteriana Mackenzie (2026)**

Autores: Karla Maria Ramos da Silva · Oscar Luz · Rafael Hessel Sichetti · Vitor Assunção Rocha

---
**Objetivo:** Desenvolver um sistema de recomendação de produtos fitness utilizando o dataset público de avaliações da Amazon (McAuley, UCSD), aplicando Filtragem Colaborativa Item-Item com Similaridade do Cosseno.

**Pipeline:**
1. Carregamento e preparação dos dados
2. Limpeza e tratamento
3. Análise exploratória (EDA)
4. Construção da matriz usuário-produto
5. Cálculo de similaridade (cosseno)
6. Geração de recomendações
7. Avaliação com métricas (Precision@K, Recall@K, HitRate@K)

## 0. Instalação de dependências

In [ ]:
# Execute este bloco apenas se necessário
# !pip install pandas numpy scikit-learn matplotlib seaborn tqdm

## 1. Importações

In [ ]:
import pandas as pd
import numpy as np
import json
import gzip
import os
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize
from scipy.sparse import csr_matrix

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})

SEED = 42
np.random.seed(SEED)
print('Bibliotecas carregadas com sucesso.')

## 2. Carregamento dos dados

O dataset está disponível em: https://cseweb.ucsd.edu/~jmcauley/datasets.html

Categoria utilizada: **Sports and Outdoors** (subconjunto fitness)

Formato dos arquivos: JSON Lines compactados (.jsonl.gz)

In [ ]:
# -------------------------------------------------------------------
# Função para carregar o dataset no formato JSON Lines (.jsonl / .jsonl.gz)
# -------------------------------------------------------------------

def load_jsonl(filepath: str, max_records: int = None) -> pd.DataFrame:
    """
    Carrega um arquivo JSON Lines (opcionalmente comprimido em .gz).
    
    Parâmetros
    ----------
    filepath    : caminho do arquivo .jsonl ou .jsonl.gz
    max_records : limite de registros (None = sem limite)
    
    Retorna
    -------
    DataFrame com os registros carregados.
    """
    records = []
    open_fn = gzip.open if filepath.endswith('.gz') else open
    with open_fn(filepath, 'rt', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if max_records and i >= max_records:
                break
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError:
                continue
    return pd.DataFrame(records)


# -------------------------------------------------------------------
# SUBSTITUA o caminho abaixo pelo caminho real do arquivo baixado
# Exemplo: 'data/Sports_and_Outdoors.jsonl.gz'
# -------------------------------------------------------------------
REVIEWS_PATH = 'Sports_and_Outdoors.jsonl.gz'
META_PATH    = 'meta_Sports_and_Outdoors.jsonl.gz'  # metadados (opcional)

if Path(REVIEWS_PATH).exists():
    print('Carregando dataset de avaliações...')
    df_raw = load_jsonl(REVIEWS_PATH, max_records=500_000)
    print(f'Registros carregados: {len(df_raw):,}')
    print(df_raw.head(3))
else:
    print(f'Arquivo não encontrado: {REVIEWS_PATH}')
    print('Gerando dataset sintético para demonstração...')

    # Dataset sintético para demonstração sem o arquivo real
    rng = np.random.default_rng(SEED)
    n_users, n_items = 500, 80
    user_ids  = [f'U{i:04d}' for i in range(n_users)]
    item_ids  = [f'B{i:07d}' for i in range(n_items)]
    
    product_names = [
        'Halteres ajustáveis 20kg', 'Tapete de yoga antiderrapante',
        'Proteína whey 2kg baunilha', 'Corda de pular speed',
        'Faixa elástica resistência', 'Luvas de treino profissional',
        'Monitor de frequência cardíaca', 'Shaker de proteína 700ml',
        'Bicicleta ergométrica dobrável', 'Kettlebell de ferro fundido 12kg',
        'Colchonete de ginástica 10mm', 'Tênis de corrida masculino',
        'Tênis de corrida feminino', 'Suplemento BCAA 300g',
        'Creatina monohidratada 300g', 'Barra de pull-up porta',
        'Rolo de espuma massagem', 'Meia de compressão esportiva',
        'Squeeze de alumínio 750ml', 'Esteira elétrica dobrável',
    ] + [f'Produto fitness {i}' for i in range(20, n_items)]
    
    categories = [
        'Equipamentos', 'Acessórios', 'Suplementos', 'Vestuário', 'Eletrônicos'
    ]
    item_cats = rng.choice(categories, size=n_items)
    
    # Simula preferências por cluster de usuário
    rows = []
    for u in user_ids:
        n_reviews = rng.integers(3, 18)
        items = rng.choice(item_ids, size=n_reviews, replace=False)
        for asin in items:
            rating = int(rng.choice([1,2,3,4,5], p=[0.06,0.06,0.12,0.26,0.50]))
            rows.append({'user_id': u, 'asin': asin, 'rating': rating,
                         'timestamp': int(pd.Timestamp('2022-01-01').timestamp()) + int(rng.integers(0, 86400*365*2))})
    
    df_raw = pd.DataFrame(rows)
    
    # Cria lookup de metadados
    df_meta = pd.DataFrame({'asin': item_ids,
                             'title': product_names[:n_items],
                             'category': item_cats})
    print(f'Dataset sintético criado: {len(df_raw):,} avaliações · {n_users} usuários · {n_items} produtos')

## 3. Limpeza e pré-processamento

In [ ]:
# Mapeamento de colunas do dataset real da Amazon (ajuste se necessário)
COLUMN_MAP = {
    'reviewerID': 'user_id',
    'asin':       'asin',
    'overall':    'rating',
    'unixReviewTime': 'timestamp',
    'reviewText': 'review_text',
    'summary':    'summary',
}

def preprocess(df: pd.DataFrame) -> pd.DataFrame:
    """
    Renomeia colunas, descarta duplicatas e trata valores ausentes.
    """
    # Renomeia se as colunas originais existirem
    rename = {k: v for k, v in COLUMN_MAP.items() if k in df.columns}
    df = df.rename(columns=rename)
    
    required = ['user_id', 'asin', 'rating']
    missing_cols = [c for c in required if c not in df.columns]
    if missing_cols:
        raise ValueError(f'Colunas ausentes: {missing_cols}')
    
    # Mantém apenas colunas relevantes
    keep = [c for c in required + ['timestamp', 'review_text', 'summary'] if c in df.columns]
    df = df[keep].copy()
    
    # Remove nulos nas colunas essenciais
    before = len(df)
    df.dropna(subset=required, inplace=True)
    print(f'Registros removidos por nulos: {before - len(df):,}')
    
    # Converte rating para inteiro e filtra valores válidos
    df['rating'] = pd.to_numeric(df['rating'], errors='coerce').astype('Int64')
    df = df[df['rating'].between(1, 5)]
    
    # Remove duplicatas (mesmo usuário + mesmo produto — mantém a mais recente)
    if 'timestamp' in df.columns:
        df.sort_values('timestamp', ascending=False, inplace=True)
    df.drop_duplicates(subset=['user_id', 'asin'], keep='first', inplace=True)
    
    df.reset_index(drop=True, inplace=True)
    return df


df = preprocess(df_raw)

print(f'\nDataset limpo:')
print(f'  Avaliações : {len(df):,}')
print(f'  Usuários   : {df["user_id"].nunique():,}')
print(f'  Produtos   : {df["asin"].nunique():,}')
print(f'  Esparsidade: {1 - len(df) / (df["user_id"].nunique() * df["asin"].nunique()):.2%}')
df.head()

### 3.1 Filtragem por frequência mínima
Usuários e produtos com poucas interações aumentam a esparsidade e degradam a qualidade das recomendações.

In [ ]:
MIN_USER_RATINGS = 5   # usuário deve ter avaliado ao menos N produtos
MIN_ITEM_RATINGS = 5   # produto deve ter recebido ao menos N avaliações

def filter_by_frequency(df, min_user=5, min_item=5):
    before = len(df)
    for _ in range(10):  # iterações para convergência
        user_counts = df['user_id'].value_counts()
        item_counts = df['asin'].value_counts()
        df = df[df['user_id'].isin(user_counts[user_counts >= min_user].index)]
        df = df[df['asin'].isin(item_counts[item_counts >= min_item].index)]
        if len(df) == before:
            break
        before = len(df)
    return df.reset_index(drop=True)

df_filtered = filter_by_frequency(df, MIN_USER_RATINGS, MIN_ITEM_RATINGS)

print('Após filtragem por frequência mínima:')
print(f'  Avaliações : {len(df_filtered):,}')
print(f'  Usuários   : {df_filtered["user_id"].nunique():,}')
print(f'  Produtos   : {df_filtered["asin"].nunique():,}')
print(f'  Esparsidade: {1 - len(df_filtered) / (df_filtered["user_id"].nunique() * df_filtered["asin"].nunique()):.2%}')

## 4. Análise Exploratória dos Dados (EDA)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Distribuição de ratings
rating_counts = df_filtered['rating'].value_counts().sort_index()
axes[0].bar(rating_counts.index, rating_counts.values,
            color=['#F09595','#FAC775','#B5D4F4','#9FE1CB','#5DCAA5'], edgecolor='none')
axes[0].set_title('Distribuição de ratings', fontsize=12)
axes[0].set_xlabel('Estrelas')
axes[0].set_ylabel('Quantidade')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

# Avaliações por usuário
user_counts = df_filtered['user_id'].value_counts()
axes[1].hist(user_counts.values, bins=30, color='#85B7EB', edgecolor='none')
axes[1].set_title('Avaliações por usuário', fontsize=12)
axes[1].set_xlabel('N.º de avaliações')
axes[1].set_ylabel('Frequência')

# Avaliações por produto
item_counts = df_filtered['asin'].value_counts()
axes[2].hist(item_counts.values, bins=30, color='#5DCAA5', edgecolor='none')
axes[2].set_title('Avaliações por produto', fontsize=12)
axes[2].set_xlabel('N.º de avaliações')
axes[2].set_ylabel('Frequência')

plt.tight_layout()
plt.savefig('eda_distribuicoes.png', bbox_inches='tight')
plt.show()

print(f'Rating médio geral : {df_filtered["rating"].mean():.2f}')
print(f'Mediana de av./usuário: {user_counts.median():.0f}')
print(f'Mediana de av./produto: {item_counts.median():.0f}')

In [ ]:
# Top 15 produtos mais avaliados
top_items = df_filtered['asin'].value_counts().head(15)

# Tenta enriquecer com nomes se df_meta existir
if 'df_meta' in dir():
    meta_lookup = df_meta.set_index('asin')['title'].to_dict()
    labels = [meta_lookup.get(a, a) for a in top_items.index]
else:
    labels = top_items.index.tolist()

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(range(len(top_items)), top_items.values[::-1], color='#5DCAA5', edgecolor='none')
ax.set_yticks(range(len(top_items)))
ax.set_yticklabels(labels[::-1], fontsize=9)
ax.set_xlabel('N.º de avaliações')
ax.set_title('Top 15 produtos mais avaliados', fontsize=12)
plt.tight_layout()
plt.savefig('top15_produtos.png', bbox_inches='tight')
plt.show()

## 5. Construção da matriz usuário-produto

In [ ]:
# Codifica usuários e produtos como índices inteiros
user_encoder = {u: i for i, u in enumerate(df_filtered['user_id'].unique())}
item_encoder = {a: i for i, a in enumerate(df_filtered['asin'].unique())}
item_decoder = {i: a for a, i in item_encoder.items()}

df_filtered['user_idx'] = df_filtered['user_id'].map(user_encoder)
df_filtered['item_idx'] = df_filtered['asin'].map(item_encoder)

n_users = len(user_encoder)
n_items = len(item_encoder)

# Constrói matriz esparsa (usuário × produto)
user_item_matrix = csr_matrix(
    (df_filtered['rating'].astype(float),
     (df_filtered['user_idx'], df_filtered['item_idx'])),
    shape=(n_users, n_items)
)

print(f'Matriz usuário-produto: {n_users} × {n_items}')
print(f'Esparsidade: {1 - user_item_matrix.nnz / (n_users * n_items):.2%}')

## 6. Cálculo de similaridade do cosseno (item-item)

In [ ]:
# Transpõe para item × usuário (cada linha = vetor do item)
item_user_matrix = user_item_matrix.T

# Calcula similaridade do cosseno entre todos os pares de itens
print('Calculando similaridade do cosseno...')
item_similarity = cosine_similarity(item_user_matrix, dense_output=False)
print(f'Matriz de similaridade: {item_similarity.shape}')

# Visualiza sub-matriz para os 20 primeiros itens
top_n = min(20, n_items)
sim_dense = item_similarity[:top_n, :top_n].toarray()

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(sim_dense, ax=ax, cmap='Blues', vmin=0, vmax=1,
            linewidths=0.3, linecolor='white',
            xticklabels=[item_decoder[i] for i in range(top_n)],
            yticklabels=[item_decoder[i] for i in range(top_n)])
ax.set_title(f'Mapa de calor — similaridade entre os primeiros {top_n} produtos', fontsize=12)
plt.xticks(fontsize=7, rotation=45, ha='right')
plt.yticks(fontsize=7)
plt.tight_layout()
plt.savefig('heatmap_similaridade.png', bbox_inches='tight')
plt.show()

## 7. Geração de recomendações

In [ ]:
def get_similar_items(asin: str, top_k: int = 10) -> pd.DataFrame:
    """
    Retorna os top_k itens mais similares ao produto indicado.
    
    Parâmetros
    ----------
    asin  : código do produto de referência
    top_k : quantidade de recomendações
    
    Retorna
    -------
    DataFrame com asin, similaridade (cosseno) e nome do produto (se disponível).
    """
    if asin not in item_encoder:
        raise ValueError(f'Produto não encontrado: {asin}')
    
    idx = item_encoder[asin]
    sim_scores = np.array(item_similarity[idx].todense()).flatten()
    sim_scores[idx] = 0  # exclui o próprio item
    
    top_indices = np.argsort(sim_scores)[::-1][:top_k]
    results = pd.DataFrame({
        'asin':        [item_decoder[i] for i in top_indices],
        'similaridade': sim_scores[top_indices].round(4)
    })
    
    if 'df_meta' in dir():
        meta_lookup = df_meta.set_index('asin')['title'].to_dict()
        results['nome'] = results['asin'].map(meta_lookup).fillna('—')
    
    return results


def recommend_for_user(user_id: str, top_k: int = 10) -> pd.DataFrame:
    """
    Gera recomendações para um usuário específico usando filtragem colaborativa item-item.
    
    Estratégia: para cada item avaliado positivamente (rating >= 4), busca itens similares
    ainda não vistos pelo usuário, ponderados pela nota dada.
    """
    if user_id not in user_encoder:
        raise ValueError(f'Usuário não encontrado: {user_id}')
    
    u_idx = user_encoder[user_id]
    user_row = user_item_matrix[u_idx].toarray().flatten()
    seen_items = set(np.where(user_row > 0)[0])
    
    scores = np.zeros(n_items)
    liked = np.where(user_row >= 4)[0]
    
    for i in liked:
        weight = user_row[i]
        sims = np.array(item_similarity[i].todense()).flatten()
        scores += weight * sims
    
    # Zera itens já vistos
    scores[list(seen_items)] = 0
    
    top_indices = np.argsort(scores)[::-1][:top_k]
    results = pd.DataFrame({
        'asin':  [item_decoder[i] for i in top_indices],
        'score': scores[top_indices].round(4)
    })
    
    if 'df_meta' in dir():
        meta_lookup = df_meta.set_index('asin')['title'].to_dict()
        results['nome'] = results['asin'].map(meta_lookup).fillna('—')
    
    return results


# Demonstração — item mais avaliado
sample_asin = df_filtered['asin'].value_counts().index[0]
print(f'Produto de referência: {sample_asin}\n')
print('Itens mais similares:')
get_similar_items(sample_asin, top_k=5)

In [ ]:
# Demonstração — recomendações para um usuário
sample_user = df_filtered['user_id'].value_counts().index[0]
print(f'Recomendações para o usuário: {sample_user}\n')
recommend_for_user(sample_user, top_k=10)

## 8. Avaliação do modelo

Métricas utilizadas:
- **Precision@K**: fração dos K itens recomendados que são relevantes
- **Recall@K**: fração dos itens relevantes que aparecem nos K recomendados
- **HitRate@K**: proporção de usuários para os quais ao menos 1 item relevante aparece nos K recomendados

In [ ]:
from sklearn.model_selection import train_test_split

# Divide interações: 80% treino / 20% teste (por usuário)
train_list, test_list = [], []
for user_id, group in df_filtered.groupby('user_id'):
    if len(group) < 4:
        train_list.append(group)
        continue
    tr, te = train_test_split(group, test_size=0.2, random_state=SEED)
    train_list.append(tr)
    test_list.append(te)

df_train = pd.concat(train_list).reset_index(drop=True)
df_test  = pd.concat(test_list).reset_index(drop=True)

print(f'Treino: {len(df_train):,} avaliações | Teste: {len(df_test):,} avaliações')

# Reconstrói a matriz e a similaridade apenas com o conjunto de treino
train_matrix = csr_matrix(
    (df_train['rating'].astype(float),
     (df_train['user_idx'], df_train['item_idx'])),
    shape=(n_users, n_items)
)
item_sim_train = cosine_similarity(train_matrix.T, dense_output=False)
print('Matriz de similaridade (treino) calculada.')

In [ ]:
def precision_at_k(recommended: list, relevant: set, k: int) -> float:
    recommended_k = recommended[:k]
    hits = len(set(recommended_k) & relevant)
    return hits / k if k > 0 else 0.0

def recall_at_k(recommended: list, relevant: set, k: int) -> float:
    recommended_k = recommended[:k]
    hits = len(set(recommended_k) & relevant)
    return hits / len(relevant) if relevant else 0.0

def hit_rate_at_k(recommended: list, relevant: set, k: int) -> float:
    return 1.0 if set(recommended[:k]) & relevant else 0.0


def evaluate(df_train, df_test, item_sim, K=10, relevance_threshold=4, max_users=200):
    """
    Avalia o modelo calculando Precision@K, Recall@K e HitRate@K
    para uma amostra de usuários do conjunto de teste.
    """
    # Agrupa itens relevantes do teste por usuário
    test_relevant = (
        df_test[df_test['rating'] >= relevance_threshold]
        .groupby('user_id')['asin'].apply(set).to_dict()
    )
    
    train_seen = df_train.groupby('user_id')['asin'].apply(set).to_dict()
    
    eval_users = [u for u in test_relevant if u in user_encoder][:max_users]
    
    precisions, recalls, hitrates = [], [], []
    
    for user_id in eval_users:
        u_idx = user_encoder[user_id]
        user_row = train_matrix[u_idx].toarray().flatten()
        seen = set(np.where(user_row > 0)[0])
        liked = np.where(user_row >= relevance_threshold)[0]
        
        if len(liked) == 0:
            continue
        
        scores = np.zeros(n_items)
        for i in liked:
            scores += user_row[i] * np.array(item_sim[i].todense()).flatten()
        scores[list(seen)] = 0
        
        top_k_idx = np.argsort(scores)[::-1][:K]
        recommended = [item_decoder[i] for i in top_k_idx]
        relevant    = test_relevant[user_id]
        
        precisions.append(precision_at_k(recommended, relevant, K))
        recalls.append(recall_at_k(recommended, relevant, K))
        hitrates.append(hit_rate_at_k(recommended, relevant, K))
    
    return {
        f'Precision@{K}': np.mean(precisions),
        f'Recall@{K}':    np.mean(recalls),
        f'HitRate@{K}':   np.mean(hitrates),
        'n_users_avaliados': len(precisions)
    }


print('Avaliando modelo...')
results = evaluate(df_train, df_test, item_sim_train, K=10)
for k, v in results.items():
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')

In [ ]:
# Gráfico comparativo — baseline (valores do relatório) vs modelo atual
baseline = {'Precision@10': 0.0040, 'Recall@10': 0.0400, 'HitRate@10': 0.0400}
current  = {k: results[k] for k in baseline}

labels = list(baseline.keys())
x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 4))
bars1 = ax.bar(x - width/2, list(baseline.values()), width, label='Baseline (TF-IDF)', color='#B4B2A9', edgecolor='none')
bars2 = ax.bar(x + width/2, list(current.values()),  width, label='Modelo atual (CF item-item)', color='#5DCAA5', edgecolor='none')

ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=11)
ax.set_ylabel('Valor da métrica')
ax.set_title('Comparativo de métricas: baseline vs modelo aprimorado', fontsize=12)
ax.legend()
ax.bar_label(bars1, fmt='%.4f', fontsize=8, padding=3)
ax.bar_label(bars2, fmt='%.4f', fontsize=8, padding=3)
plt.tight_layout()
plt.savefig('metricas_comparativo.png', bbox_inches='tight')
plt.show()

## 9. Conclusões e próximos passos

O modelo baseline de Filtragem Colaborativa Item-Item com similaridade do cosseno foi implementado com sucesso.

**Limitações identificadas:**
- Alta esparsidade da matriz usuário-produto
- Problema de cold-start para novos usuários e produtos
- Representação de perfil de usuário simplificada (média dos vetores)

**Próximos passos:**
- Fatoração de matriz (SVD / ALS) para lidar com a esparsidade
- Análise de sentimentos nos textos das avaliações (VADER, BERT)
- Abordagem híbrida: colaborativa + baseada em conteúdo
- Ajuste fino de hiperparâmetros (k vizinhos, limiar de relevância)
- Avaliação com NDCG@K e MAP@K

## Referências

- McAuley, J. *Amazon product data*. UCSD. https://cseweb.ucsd.edu/~jmcauley/datasets.html
- Adomavicius, G.; Tuzhilin, A. (2005). Toward the next generation of recommender systems. *IEEE TKDE*, 17(6).
- Ricci, F.; Rokach, L.; Shapira, B. (2011). *Recommender Systems Handbook*. Springer.
- Cremonesi, P.; Koren, Y.; Turrin, R. (2010). Performance of recommender algorithms on top-N recommendation tasks. *ACM RecSys*.
- Alamdari, P. M. et al. (2020). A systematic study on the recommender systems in e-commerce. *IEEE Access*, 8.